# 02 — Baseline end-to-end (braindecode)
**EEGNet, ShallowFBCSPNet, Deep4Net** su segnale grezzo, subject-dependent su 15 soggetti.
Early stopping sul Validation set, valutazione sul Test set (true label da answer sheet).

> Metti `use_wandb=True` dopo `pip install wandb && wandb login` per loggare su W&B (entity `uras-daniele22-politecnico-di-milano`, project `miralis-imagined-speech`, tag `track3`).

In [ ]:
# --- setup: rende importabili i moduli track3_*.py ---
import sys, os
sys.path.insert(0, os.path.abspath('.'))
import numpy as np, matplotlib.pyplot as plt
import track3_config as C, track3_io as io, track3_preproc as P
print(C.summary())
assert C.DATA_ROOT is not None, C._no_data_msg()
import track3_train as T
import track3_models as M
device=C.get_device(); print('device:', device)

## 1. EEGNet su tutti i soggetti
⚠️ Sulla GPU della VM è veloce; su CPU può richiedere qualche minuto per modello.

In [ ]:
df_eeg, res_eeg = T.run_subject_dependent(
    'eegnet', use_wandb=False,
    train_kwargs=dict(epochs=200, patience=30, lr=1e-3, batch_size=32))
T.save_metrics(df_eeg, 'eegnet'); df_eeg

In [ ]:
T.plot_per_subject(df_eeg, 'eegnet'); plt.show()
T.plot_confusion(res_eeg, model_name='eegnet'); plt.show()

## 2. ShallowFBCSPNet

In [ ]:
df_sh, res_sh = T.run_subject_dependent(
    'shallow', train_kwargs=dict(epochs=200, patience=30, lr=1e-3, batch_size=32))
T.save_metrics(df_sh, 'shallow')
T.plot_per_subject(df_sh, 'shallow'); plt.show()

## 3. Deep4Net

In [ ]:
df_d4, res_d4 = T.run_subject_dependent(
    'deep4', train_kwargs=dict(epochs=200, patience=30, lr=1e-3, batch_size=32))
T.save_metrics(df_d4, 'deep4')
T.plot_per_subject(df_d4, 'deep4'); plt.show()

## 4. Riepilogo baseline

In [ ]:
import pandas as pd
summary = pd.DataFrame({
    'eegnet':  df_eeg.test_acc, 'shallow': df_sh.test_acc, 'deep4': df_d4.test_acc,
}).describe().loc[['mean','std','min','max']]
print('chance =', C.CHANCE_LEVEL); summary

---
## 5. Protocollo SUBJECT-MIXED (confronto con CBraMod, Table 9)

Il paper **CBraMod** (ICLR 2025), Table 9, riporta su questo stesso dataset (BCIC2020-3) numeri
molto più alti (EEGNet **0.44** bAcc). Il motivo **non è il preprocessing**: loro **NON** usano il
protocollo ufficiale subject-dependent. Mettono i **15 soggetti in un unico dataset** e allenano
**un solo modello**:

| | Subject-dependent (§1–4, protocollo ufficiale) | Subject-mixed (CBraMod Table 9) |
|---|---|---|
| Modelli | 15 (uno per soggetto) | **1** |
| Trial di training per modello | 300 | **~4500** (15×) |
| Test | trial di S_i col modello di S_i | trial di tutti, modello che ha già visto tutti i soggetti |
| Val per early stopping | 50 (rumoroso) | **~750** (affidabile) |

`run_subject_mixed` replica il loro protocollo. Serve per **confrontarsi con Table 9** e per avere
in tesi entrambi i numeri. La standardizzazione resta **per-soggetto** (nessun leakage).

In [ ]:
# EEGNet subject-mixed (un solo modello sui 15 soggetti pooled)
df_eeg_mix, res_eeg_mix = T.run_subject_mixed(
    'eegnet', use_wandb=False,
    train_kwargs=dict(epochs=200, patience=30, lr=1e-3, batch_size=64))
T.save_metrics(df_eeg_mix, 'eegnet_mixed')
print('\nAccuratezza per-soggetto sotto il modello mixed:')
df_eeg_mix.round(3)

In [ ]:
# (opzionale) anche shallow e deep4 in subject-mixed
df_sh_mix, _  = T.run_subject_mixed('shallow', train_kwargs=dict(epochs=200, patience=30, lr=1e-3, batch_size=64))
df_d4_mix, _  = T.run_subject_mixed('deep4',   train_kwargs=dict(epochs=200, patience=30, lr=1e-3, batch_size=64))
T.save_metrics(df_sh_mix, 'shallow_mixed'); T.save_metrics(df_d4_mix, 'deep4_mixed')

In [ ]:
# Confronto finale: nostro subject-dependent vs nostro subject-mixed vs CBraMod Table 9
import pandas as pd
cbramod_tab9 = {'eegnet': 0.4413, 'shallow': None, 'deep4': None}  # bAcc riportata nel paper (EEGNet)
compare = pd.DataFrame({
    'subject_dependent (media 15)': {
        'eegnet': df_eeg.test_bacc.mean(), 'shallow': df_sh.test_bacc.mean(), 'deep4': df_d4.test_bacc.mean()},
    'subject_mixed (1 modello)': {
        'eegnet': df_eeg_mix.loc['ALL','test_bacc'],
        'shallow': df_sh_mix.loc['ALL','test_bacc'], 'deep4': df_d4_mix.loc['ALL','test_bacc']},
    'CBraMod Table 9 (EEGNet)': cbramod_tab9,
})
print('Balanced Accuracy — chance =', C.CHANCE_LEVEL)
compare.round(3)

---
## 6. Diagnosi: perché siamo a chance anche subject-mixed?

Con 4500 trial di training e val da 750, se restiamo a chance il problema **non è il protocollo**:
il modello **non impara**. Tre sospetti, li distinguiamo qui sotto.

**Chiave di lettura — la `train_acc`** (ora stampata da ogni run):
- `train_acc` ≈ chance → il modello **non fitta nemmeno il training** = *underfitting* → bug (doppio
  softmax) o preprocessing che distrugge il segnale.
- `train_acc` alta, `test` ≈ chance → problema di **generalizzazione** (overfitting / info non trasferibile).

Test in ordine:
1. **doppio softmax** — `output_is_logprob` su un modello fresco (ora `make_braindecode` lo rimuove);
2. **c'è segnale?** — LDA classico su band-power con 3 preprocessing diversi;
3. **preprocessing** — EEGNet subject-mixed con `PP_DEFAULT` vs `PP_MINIMAL` vs `PP_HIGHGAMMA`.

In [ ]:
# --- Test 1: doppio softmax? (deve stampare False dopo il fix in make_braindecode) ---
import torch
m = M.make_braindecode('eegnet', n_chans=C.N_CHANNELS, n_times=512)
x = torch.randn(8, C.N_CHANNELS, 512)
print('output è log-prob (=> doppio softmax col CrossEntropy)?', M.output_is_logprob(m, x))
print('somma exp per riga (logits => != 1):', m(x[:3]).exp().sum(1).detach().numpy().round(2))

In [ ]:
# --- Test 2: c'è segnale decodificabile? LDA su band-power, 3 preprocessing (subject-mixed) ---
print('DEFAULT (bp 0.5-45, crop 0-2000):')
T.classical_baseline(pp_kwargs=C.PP_DEFAULT)
print('\nMINIMAL (epoca intera, nessun filtro):')
T.classical_baseline(pp_kwargs=C.PP_MINIMAL)
print('\nHIGHGAMMA (bp 0.5-100, epoca intera):')
T.classical_baseline(pp_kwargs=C.PP_HIGHGAMMA)

In [ ]:
# --- Test 3: EEGNet subject-mixed col fix, confronto preprocessing (guarda TRAIN_ACC) ---
for tag, pp in [('DEFAULT', C.PP_DEFAULT), ('MINIMAL', C.PP_MINIMAL), ('HIGHGAMMA', C.PP_HIGHGAMMA)]:
    print(f'\n===== preprocessing = {tag} =====')
    df_m, res_m = T.run_subject_mixed(
        'eegnet', pp_kwargs=pp,
        train_kwargs=dict(epochs=200, patience=30, lr=1e-3, batch_size=64))
# Interpretazione:
#  - se train_acc resta ~0.2 ovunque  -> il problema è a monte (dati/label/loss), NON il preproc
#  - se train_acc sale ma test resta basso -> overfitting/generalizzazione
#  - se HIGHGAMMA/MINIMAL migliora test -> confermato: il bandpass 0.5-45 buttava via segnale